# Building a machine learning model in python

We are going to try to use our data to build a predictive model. My example tries to do it with the sample BBC Programme data from previous examples. It reprocesses that to create a 'Genre' field alongside the Text and Description fields. It then uses those data fields to predict a genre for any programme description provided... we'll see how well it does!

In [1]:

import pandas as pd

# Load the BBC News dataset
df = pd.read_csv("BBCNewsTrain (1).csv")

# Check the dataset
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

print("\nNews categories:")
print(df["Category"].value_counts())


Dataset shape: (1490, 3)

Column names:
['ArticleId', 'Text', 'Category']

First 5 rows:


,ArticleId,Text,Category
0,1833,worldcom ex-boss launches defence lawyers defe...,business
1,154,german business confidence slides german busin...,business
2,1101,bbc poll indicates economic gloom citizens in ...,business
3,1976,lifestyle governs mobile choice faster bett...,tech
4,917,enron bosses in $168m payout eighteen former e...,business



Missing values:
ArticleId    0
Text         0
Category     0
dtype: int64

News categories:
Category
sport            346
business         336
politics         274
entertainment    273
tech             261
Name: count, dtype: int64


In [2]:

from sklearn.model_selection import train_test_split

# News text as input
X = df["Text"]

# Existing news categories as target
y = df["Category"]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training articles:", len(X_train))
print("Testing articles:", len(X_test))

print("\nTraining categories:")
print(y_train.value_counts())

print("\nTesting categories:")
print(y_test.value_counts())


Training articles: 1192
Testing articles: 298

Training categories:
Category
sport            277
business         269
politics         219
entertainment    218
tech             209
Name: count, dtype: int64

Testing categories:
Category
sport            69
business         67
entertainment    55
politics         55
tech             52
Name: count, dtype: int64


In [3]:

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

# Build a news classification model
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english"
    )),
    ("classifier", MultinomialNB())
])

# Train the model using the training dataset
model.fit(X_train, y_train)

print("Model training completed successfully!")
print("Number of training articles:", len(X_train))
print("News categories:", list(model.classes_))


Model training completed successfully!
Number of training articles: 1192
News categories: [np.str_('business'), np.str_('entertainment'), np.str_('politics'), np.str_('sport'), np.str_('tech')]


In [4]:

from sklearn.metrics import accuracy_score, classification_report

# Predict categories for the test articles
y_pred = model.predict(X_test)

# Calculate overall accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy:.2%}")

# Evaluate performance for each category
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Model Accuracy: 97.32%

Classification Report:
               precision    recall  f1-score   support

     business       0.96      0.99      0.97        67
entertainment       1.00      0.96      0.98        55
     politics       0.98      0.95      0.96        55
        sport       0.97      1.00      0.99        69
         tech       0.96      0.96      0.96        52

     accuracy                           0.97       298
    macro avg       0.97      0.97      0.97       298
 weighted avg       0.97      0.97      0.97       298



In [5]:

# Test the trained model with a new news example

new_article = """
The government has announced a new education policy.
Ministers discussed funding for schools and universities,
while opposition parties questioned the proposed changes.
"""

# Predict the news category
prediction = model.predict([new_article])

print("New article:")
print(new_article)

print("Predicted category:", prediction[0])


New article:

The government has announced a new education policy.
Ministers discussed funding for schools and universities,
while opposition parties questioned the proposed changes.

Predicted category: politics


In [ ]:
## This script reprocesses the bbc data to strip out the genre from the description and save it as a new field
## It does this as we need a predictor field to build a model. The genre is what we will be predicting
## *** ONLY USE THIS SECTION IF YOU ARE USING THE bbcprogs.csv FILE ***

#import and clean data

import pandas as pd
import re

df = pd.read_csv(
    "bbcprogs.csv",
    header=None,
    names=["title", "description"]
)

def get_genre(text):

    match = re.search(
        r"Category:\s*([^\.]+)",
        str(text)
    )

    if match:
        return match.group(1).strip()

    return "Unknown"

df["genre"] = df["description"].apply(get_genre)

print(df["genre"].value_counts())

def remove_genre(text):

    return re.sub(
        r"Category:\s*[^\.]+\.",
        "",
        str(text)
    )

df["clean_description"] = (
    df["description"]
    .apply(remove_genre)
)

df["text"] = (
    df["title"] +
    " " +
    df["clean_description"]
)

genre
Crime Drama         32
Documentary         19
Drama               18
News                 9
Sport                5
Entertainment        4
Comedy               4
Unknown              3
Nature               3
Current Affairs      2
Lifestyle            2
Music                1
Film - Rom-Com       1
Science              1
History              1
Documentary Film     1
Period Drama         1
Name: count, dtype: int64


In [ ]:
#save all to new csv
df.to_csv(
    "bbcprogs_cleaned.csv",
    index=False
)

In [ ]:
## *** START HERE IF YOU ARE USING YOUR OWN, CLEANED DATA SET ***

## This script uses the SKLearn library to start to train a model
## It uses the Text and Genre fields of our enriched BBC Progs CSV file to learn about which words in the title and description most usually fit within the different genre categories

## You can use the code here to import your data set if you don't need the data transformation above
## Put the name of your CSV file here, then you don't even need to run the above cell
## You will need to edit the code to use your column names though...

#df = pd.read_csv(
#    "yourfile.csv",
#)

#train a model
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

X = df["text"]  #change to your column name
y = df["genre"] #change to your column name

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", MultinomialNB())
])

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(
    classification_report(
        y_test,
        predictions
    )
)

                 precision    recall  f1-score   support

         Comedy       0.00      0.00      0.00         1
    Crime Drama       0.30      1.00      0.46         6
Current Affairs       0.00      0.00      0.00         1
    Documentary       1.00      0.67      0.80         3
          Drama       0.00      0.00      0.00         1
  Entertainment       0.00      0.00      0.00         1
 Film - Rom-Com       0.00      0.00      0.00         1
         Nature       0.00      0.00      0.00         2
           News       0.00      0.00      0.00         3
          Sport       0.00      0.00      0.00         1
        Unknown       0.00      0.00      0.00         2

       accuracy                           0.36        22
      macro avg       0.12      0.15      0.11        22
   weighted avg       0.22      0.36      0.23        22



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
#try some predictions
while True:

    text = input(
        "Enter a programme description (or type 'quit' to finish): "
    )

    prediction = model.predict([text])

    if text.lower() == "quit":
        print("Finished")
        break

    print(
        "Predicted genre:",
        prediction[0]
    )

Enter a programme description (or type 'quit' to finish): TRAITOR
Predicted genre: Crime Drama
Enter a programme description (or type 'quit' to finish): quit
Finished


# It's only as good as the data...

You may notice at this point that my predictive model usually predicts "crime drama"! This is because the data is small and skewed. There is far more crime drama than other genres in the data. A classic example of a biased data set producing a biased model.

To make a better model, I am going to switch to a different data set - a classic training data set called the BBC News Classification Data Set, available on Kaggle. I have provided this on Minerva as BBCNewsTrain.csv and it is available on Kaggle here: https://www.kaggle.com/competitions/learn-ai-bbc/overview

In [ ]:
#use a different model from Kaggle - BBC news classifier data set
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

df = pd.read_csv(
    "BBCNewsTrain.csv",
)

X = df["Text"]
y = df["Category"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            stop_words="english",
            max_features=5000
        )
    ),
    (
        "classifier",
        MultinomialNB()
    )
])

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(
    classification_report(
        y_test,
        predictions
    )
)

               precision    recall  f1-score   support

     business       0.96      0.97      0.96        67
entertainment       1.00      1.00      1.00        55
     politics       0.96      0.95      0.95        55
        sport       1.00      1.00      1.00        69
         tech       0.96      0.96      0.96        52

     accuracy                           0.98       298
    macro avg       0.98      0.98      0.98       298
 weighted avg       0.98      0.98      0.98       298



In [ ]:
#Type your news story here and let the model predict its category

new_text = """
Manchester United secured a dramatic
victory in the Premier League with a
late winner at Old Trafford.
"""

prediction = model.predict([new_text])

print(prediction[0])

sport
